# 01 - Conspiracy Adoption Analysis

This notebook demonstrates the full empirical analysis pipeline:
1. Data loading and network construction
2. Sequential Cox models (Models 1-3)
3. Semantic clustering with Silhouette Score optimization
4. Cognitive barrier and settler effect analysis
5. Gateway effect identification from Model 2

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from conspiracy_analysis import BOT_SCORE_THRESHOLD, EXPOSURE_WINDOW
from conspiracy_analysis.data.loader import load_graph_and_tweets, load_semantic_distance_matrix
from conspiracy_analysis.config import (
    apply_conspiracy_config_to_graph,
    filter_semantic_matrix,
    get_baseline_conspiracy,
    get_included_conspiracies,
)
from conspiracy_analysis.utils.helpers import passes_bot_filter
from conspiracy_analysis.data.preprocessing import (
    get_first_activation_data,
    get_second_activation_data,
    get_third_activation_data,
    get_nth_activation_data,
    get_max_model_number,
)
from conspiracy_analysis.analysis.semantic import (
    find_optimal_clustering,
    assign_clusters,
    save_clustering_result,
    compute_cross_cluster_flag,
    compute_temporal_distance_matrix,
    compute_peak_frequency_distance_matrix,
)
from conspiracy_analysis.analysis.statistics import (
    extract_user_timelines,
    compute_semantic_barrier_analysis,
    compute_settler_effect,
    test_barrier_significance,
    test_settler_significance,
)
from conspiracy_analysis.models.cox_adoption import fit_cox_model, fit_all_cox_models
from conspiracy_analysis.models.baseline_hazards import (
    parametrize_all_baselines,
    compute_all_decay_times,
)
from conspiracy_analysis.models.gateway_effects import identify_gateway_conspiracies
from conspiracy_analysis.models.bootstrap_inference import (
    ModelBootstrapSpec,
    run_cox_user_bootstrap,
    run_timeline_bootstrap,
    summarize_cox_bootstrap,
)
from conspiracy_analysis.visualization.figures import (
    plot_dendrogram,
    plot_cognitive_barrier,
    plot_settler_effect,
)
from conspiracy_analysis.visualization.plots import (
    plot_hazard_ratios,
    plot_baseline_hazard_fit,
    plot_silhouette_heatmap,
    plot_instantaneous_baseline_hazards,
    plot_decay_times,
)
from conspiracy_analysis.utils.helpers import get_first_appearance_times, get_peak_frequency_times

import logging
logging.basicConfig(level=logging.INFO)

# Bootstrap inference settings.
# BOOTSTRAP_N_JOBS controls joblib workers. Use -1 for all available cores, or set a positive integer.
# Cox and timeline final draw counts are configured independently.
# Example: BOOTSTRAP_N_JOBS = 20 and TIMELINE_BOOTSTRAP_FINAL_DRAWS = 60 schedules about 3 draws per worker.
BOOTSTRAP_N_JOBS = 128
BOOTSTRAP_SMOKE_DRAWS = 10
COX_BOOTSTRAP_FINAL_DRAWS = 200
TIMELINE_BOOTSTRAP_FINAL_DRAWS = 2000
BOOTSTRAP_MIN_SUCCESS_RATE = 0.8
BOOTSTRAP_MIN_SUCCESSES = int(np.ceil(COX_BOOTSTRAP_FINAL_DRAWS * BOOTSTRAP_MIN_SUCCESS_RATE))
BOOTSTRAP_SEED = 42
BOOTSTRAP_EVAL_TIMES = [1.0, 24.0, 72.0]

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

INTERMEDIATE_DIR = Path("../intermediate_files")
INTERMEDIATE_DIR.mkdir(exist_ok=True)
LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_SUCCESS_LOG = LOG_DIR / "bootstrap_successes.log"

def intermediate_path(filename):
    return INTERMEDIATE_DIR / filename

def log_bootstrap_success_summary(label, n_draws, n_jobs, successes=None, min_successes=None, extra=None):
    timestamp = datetime.now(timezone.utc).isoformat()
    common_fields = [
        f"timestamp={timestamp}",
        f"label={label}",
        f"n_draws={int(n_draws)}",
        f"n_jobs={int(n_jobs)}",
    ]
    if min_successes is not None:
        common_fields.append(f"min_successes={int(min_successes)}")
    if extra:
        common_fields.extend(f"{key}={value}" for key, value in extra.items())

    lines = []
    if successes:
        for name, count in sorted(successes.items()):
            count = int(count)
            fields = common_fields + [
                f"unit={name}",
                f"n_success={count}",
                f"success_rate={count / n_draws:.4f}",
            ]
            if min_successes is not None:
                fields.append(f"threshold_status={'pass' if count >= min_successes else 'fail'}")
            lines.append("Bootstrap success summary | " + " | ".join(fields))
    else:
        fields = common_fields + [
            f"n_success={int(n_draws)}",
            "success_rate=1.0000",
        ]
        lines.append("Bootstrap success summary | " + " | ".join(fields))

    with BOOTSTRAP_SUCCESS_LOG.open("a", encoding="utf-8") as handle:
        handle.write("\n".join(lines) + "\n")
    print(f"Bootstrap success summary written to {BOOTSTRAP_SUCCESS_LOG.resolve()}")


In [ ]:
# Publication style (applies rcParams globally)
from conspiracy_analysis.visualization import set_nhb_style
set_nhb_style()

## 1. Load Data

In [ ]:
import networkx as nx

# Every public analysis starts from the published anonymized graph.
G, _ = load_graph_and_tweets(from_joblib=True, time_resolution='Hour')

# Convert to undirected graph and remove isolated nodes (matches original pipeline)
G = G.to_undirected()
isolates = list(nx.isolates(G))
G.remove_nodes_from(isolates)
print(f"Removed {len(isolates)} isolated nodes")

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Conspiracies: {G.graph['conspiracy_cols']}")

In [ ]:
# Load semantic distance matrix
df_semantic = load_semantic_distance_matrix()
print(f"Semantic matrix shape: {df_semantic.shape}")
print(f"Conspiracies in semantic matrix: {list(df_semantic.index)}")

## 2. Configuration

In [ ]:
MODE = 'HUMAN'
STEP = 8  # 8-hour discretization for Cox models
EXPOSURE_WINDOW_HOURS = EXPOSURE_WINDOW  # 14 days or 336 h for counting active neighbors
PENALIZER = 0  # L2 regularization (0 = none)
CORRECTION_FOR_CENSORED = False
ATTRITION_CORRECTION = True  # Censor non-adopters at last_active_time + 1

# Reference and analytic categories come from config/conspiracies.yaml.
BASELINE_CONSPIRACY = get_baseline_conspiracy()
INCLUDED_CONSPIRACIES = get_included_conspiracies()


In [ ]:
# Apply configured conspiracy set to graph and semantic matrix.
G = apply_conspiracy_config_to_graph(G)
df_semantic = filter_semantic_matrix(df_semantic)
print(f"Using {len(G.graph['conspiracy_cols'])} conspiracies: {G.graph['conspiracy_cols']}")


## 2a. Tie-Frequency Diagnostic

Quantify how often simultaneous adoptions occur — i.e., two or more conspiracies
share the exact same first-adoption timestamp for a user. These ties affect
adoption ordering, gateway assignment, and the 0.1h nudge in Models 2+.

In [ ]:
# Tie-frequency diagnostic: how often do simultaneous adoptions occur?
conspiracies = G.graph['conspiracy_cols']

tie_step_counts = {}  # step_label -> count of users with a tie at that step
users_with_any_tie = 0
total_qualifying_users = 0

for node in G.nodes:
    if not passes_bot_filter(G, node, BOT_SCORE_THRESHOLD, MODE):
        continue

    # Get first-adoption time for each conspiracy (min of activation list)
    first_times = {}
    for c in conspiracies:
        activations = G.nodes[node].get(c, [])
        if activations:
            first_times[c] = min(activations)

    if len(first_times) < 2:
        continue  # need at least 3 adoptions to have a tie of 3

    total_qualifying_users += 1
    sorted_times = sorted(first_times.values())

    user_has_tie = False
    for i in range(1, len(sorted_times)):
        step_label = f"{i}→{i+1}"  # e.g., "1→2" for 1st vs 2nd adoption
        if sorted_times[i] == sorted_times[i - 1]:# and sorted_times[i-1]==sorted_times[i-2]:
            tie_step_counts[step_label] = tie_step_counts.get(step_label, 0) + 1
            user_has_tie = True

    if user_has_tie:
        users_with_any_tie += 1

# Report
print(f"Users with >= 2 adoptions: {total_qualifying_users}")
print(f"Users with any simultaneous adoption tie: {users_with_any_tie} "
      f"({100 * users_with_any_tie / total_qualifying_users:.1f}%)")
print(f"\nPer-step tie counts (consecutive adoptions sharing the same timestamp):")
for step in sorted(tie_step_counts.keys(), key=lambda s: int(s.split('→')[0])):
    count = tie_step_counts[step]
    print(f"  Step {step}: {count} users")


## 2b. Descriptive Overview: Conspiracies Per User

In [ ]:
# Count distinct conspiracies adopted per user (non-empty activation list)
conspiracies = G.graph['conspiracy_cols']
user_counts = {}
for node in G.nodes:
    if not passes_bot_filter(G, node, BOT_SCORE_THRESHOLD, MODE):
        continue
    count = sum(1 for c in conspiracies if G.nodes[node].get(c, []))
    if count > 0:
        user_counts[node] = count

counts_series = pd.Series(user_counts)

# Print summary statistics
print(f"Users who adopted >= 1 conspiracy: {len(counts_series)}")
print(f"Mean conspiracies per user: {counts_series.mean():.2f}")
print(f"Median: {counts_series.median():.0f}")
print(f"Max: {counts_series.max()}")
print(f"\nDistribution:")
print(counts_series.value_counts().sort_index())

# Plot histogram
fig, ax = plt.subplots(figsize=(8, 5))
bins = np.arange(0.5, counts_series.max() + 1.5, 1)
ax.hist(counts_series, bins=bins, edgecolor='black', alpha=0.8, color='#4A90E2')
ax.set_xlabel('Number of Distinct Conspiracies Shared')
ax.set_ylabel('Number of Users')
ax.set_title('Distribution of Conspiracy Breadth Per User')
ax.set_xticks(range(1, int(counts_series.max()) + 1))
fig.tight_layout()
plt.show()

## 3. Semantic Clustering (Silhouette-Optimized)

In [ ]:
conspiracies = G.graph['conspiracy_cols']

clustering_result = find_optimal_clustering(
    distance_matrix=df_semantic,
    conspiracies=conspiracies,
    methods=['ward', 'average', 'complete', 'single'],
    k_range=range(3, 9),
)

save_clustering_result(clustering_result)
print("Saved semantic_clustering.pkl")

print(f"Best method: {clustering_result['best_method']}")
print(f"Best k: {clustering_result['best_k']}")
print(f"Silhouette Score: {clustering_result['best_score']:.4f}")
print(f"\nCluster assignments:")
for cid, members in sorted(clustering_result['cluster_map'].items()):
    print(f"  Cluster {cid}: {', '.join(members)}")

In [ ]:
# Visualize silhouette scores across all configurations
plot_silhouette_heatmap(clustering_result['all_scores'])
plt.show()

In [ ]:
# Dendrogram
plot_dendrogram(
    clustering_result['linkage_matrix'],
    clustering_result['conspiracies'],
    title='Semantic Distance Dendrogram',
)
plt.show()

### 3b. Mantel Test: Semantic Distance vs. Co-Adoption Distance

In [ ]:
from conspiracy_analysis.analysis.statistics import compute_coadoption_matrix, mantel_test

# Compute empirical co-adoption similarity (Jaccard), then convert to distance
coadoption_similarity = compute_coadoption_matrix(
    G, conspiracies=conspiracies,
    bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE,
)
coadoption_distance = 1.0 - coadoption_similarity

# Mantel test: semantic distance vs. co-adoption distance.
# Direction is pre-specified: semantically similar conspiracies (small
# df_semantic) are expected to also be co-adopted by overlapping user sets
# (small coadoption_distance). Expected correlation: positive. Use the
# 'greater' one-sided test consistent with this directional hypothesis.
mantel_result = mantel_test(
    df_semantic, coadoption_distance,
    n_permutations=9999, method='spearman', seed=42,
    alternative='greater',
)
print(f"Mantel test (Spearman, one-sided 'greater'): r = {mantel_result['correlation']:.4f}, "
      f"p = {mantel_result['p_value']:.4f} "
      f"({mantel_result['n_permutations']} permutations)")

## 4. Cox Models

In [ ]:
cluster_assignments = assign_clusters(clustering_result)

### Model 1: First Conspiracy Adoption

In [ ]:
df_short_1 = get_first_activation_data(
    G, bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE, active_user_correction=True,
    attrition_correction=ATTRITION_CORRECTION,
)
print(f"Model 1 short-form: {len(df_short_1)} rows, {df_short_1['event'].sum()} events")

### Model 2: Second Conspiracy Adoption

In [ ]:
df_short_2 = get_second_activation_data(
    G, bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE,
    attrition_correction=ATTRITION_CORRECTION,
)
# Add cross_cluster flag using optimal clustering
df_short_2 = compute_cross_cluster_flag(df_short_2, G, cluster_assignments, model_number=2)
print(f"Model 2 short-form: {len(df_short_2)} rows, {df_short_2['event'].sum()} events")
print(f"Cross-cluster fraction: {df_short_2['cross_cluster'].mean():.3f}")

### Model 3+: Third Conspiracy Adoption

In [ ]:
df_short_3 = get_third_activation_data(
    G, bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE,
    attrition_correction=ATTRITION_CORRECTION,
)
# Add cross_cluster flag (no prior conspiracy flags for Model 3+)
df_short_3 = compute_cross_cluster_flag(df_short_3, G, cluster_assignments, model_number=3)
print(f"Model 3 short-form: {len(df_short_3)} rows, {df_short_3['event'].sum()} events")
print(f"Cross-cluster fraction: {df_short_3['cross_cluster'].mean():.3f}")

### Fit All Models

In [ ]:
# Clean up stale Cox model pkls from prior runs.
#
# Ensures downstream notebooks (02_simulation, 03_main_text_figures,
# 03_appendix_text_figures) never silently load stale pkls when the current run
# fits fewer or differently named models than a previous run. The pattern
# 'model_*_cox.pkl' matches every Cox pkl we write while leaving
# simulation_*.pkl and other artifacts untouched.
_cox_pkl_patterns = ['model_*_cox.pkl']
_removed = []
for _pattern in _cox_pkl_patterns:
    for _path in INTERMEDIATE_DIR.glob(_pattern):
        try:
            _path.unlink()
            _removed.append(_path.name)
        except OSError as _err:
            print(f'  could not remove {_path}: {_err}')

if _removed:
    print(f'Removed {len(_removed)} stale Cox pkl(s) before fitting:')
    for _name in sorted(_removed):
        print(f'  * {_name}')
else:
    print('No stale Cox pkls to remove.')

del _cox_pkl_patterns, _removed


In [ ]:
# Build short-form data dict for Models 1-3
short_form_data = {
    "model_1": df_short_1,
    "model_2a": df_short_2,   # conspiracy dummies (what is adopted second)
    "model_2b": df_short_2,   # first_conspiracy dummies (gateway effect)
    "model_3": df_short_3,
}

# Fit all Cox models
cox_results = fit_all_cox_models(
    short_form_data, G,
    step=STEP, penalizer=PENALIZER,
    correction_for_censored=CORRECTION_FOR_CENSORED,
    exposure_window=EXPOSURE_WINDOW_HOURS,
    baseline=BASELINE_CONSPIRACY,
)

for name, (ctv, df_long) in cox_results.items():
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    ctv.print_summary()

In [ ]:
# Save fitted models
for name, (ctv, _) in cox_results.items():
    joblib.dump(ctv, intermediate_path(f'{name}_cox.pkl'))
    print(f"Saved {name}_cox.pkl")

### Hazard Ratio Plots

In [ ]:
for name, (ctv, _) in cox_results.items():
    plot_hazard_ratios(ctv, covariate_prefix='s7_', title=f'{name}: Exposure Hazard Ratios')
    plt.show()

## 5. Baseline Hazard Parametrization

In [ ]:
# Parametrize baselines (Linear for Model 1, Weibull for Models 2+)
cox_models_dict = {name: ctv for name, (ctv, _) in cox_results.items()}
baselines = parametrize_all_baselines(cox_models_dict)

for name, params in baselines.items():
    print(f"\n{name}: type={params['type']}")
    if params['type'] == 'linear':
        print(f"  slope (a) = {params['slope']:.6e}")
    else:
        print(f"  shape (k) = {params['shape']:.4f}")
        print(f"  scale (lambda) = {params['scale']:.4f}")
    print(f"  RMSE = {params['rmse']:.6e}")

In [ ]:
# Plot baseline fits
for name, params in baselines.items():
    plot_baseline_hazard_fit(params, title=f'{name}: Baseline Hazard Fit')
    plt.show()

## 5b. Extended Models (4th, 5th, ... N-th Adoption)

Auto-detect the maximum model number from the data, fit Cox models for each
additional adoption, and analyze baseline hazard decay dynamics.

In [ ]:
# Auto-detect maximum model number from data
max_n, event_counts = get_max_model_number(
    G, min_events=1000,
    bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE,
)
# Cap at Model 10 to avoid convergence errors from sparse higher-order data
max_n = min(max_n, 10)
print(f"Maximum model number with >= 1000 events: {max_n}")
print(f"\nEvent counts by model:")
for n, count in sorted(event_counts.items()):
    if count < 1000:
        marker = " (BELOW THRESHOLD)"
    elif n == max_n:
        marker = " <-- max model"
    else:
        marker = ""
    print(f"  Model {n}: {count} events{marker}")

In [ ]:
# Build short-form data for Models 4..max_n (with cross_cluster flag)
extended_short_form = {}
for n in range(4, max_n + 1):
    print(f"Building short-form data for Model {n}...")
    df_short_n = get_nth_activation_data(
        G, n=n, bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE,
        attrition_correction=ATTRITION_CORRECTION,
    )
    df_short_n = compute_cross_cluster_flag(df_short_n, G, cluster_assignments, model_number=n)
    extended_short_form[f"model_{n}"] = df_short_n
    n_events = df_short_n["event"].sum() if not df_short_n.empty else 0
    print(f"  Model {n}: {len(df_short_n)} rows, {n_events} events")

In [ ]:
# Fit extended Cox models (Models 4..max_n) together with Models 1-3
all_short_form = {
    "model_1": df_short_1,
    "model_2a": df_short_2,
    "model_2b": df_short_2,
    "model_3": df_short_3,
    **extended_short_form,
}

all_cox_results = fit_all_cox_models(
    all_short_form, G,
    step=STEP, penalizer=PENALIZER,
    correction_for_censored=CORRECTION_FOR_CENSORED,
    exposure_window=EXPOSURE_WINDOW_HOURS,
    baseline=BASELINE_CONSPIRACY,
)

# Update cox_results to include all models
cox_results = all_cox_results

for name, (ctv, df_long) in sorted(all_cox_results.items()):
    n_events = df_long["event"].sum()
    print(f"{name}: {n_events} events in long-form, {len(ctv.params_)} parameters")

In [ ]:
# Save ALL fitted models (including Models 4 & 5) to .pkl files
for name, (ctv, _) in cox_results.items():
    joblib.dump(ctv, intermediate_path(f'{name}_cox.pkl'))
    print(f"Saved {name}_cox.pkl")

In [ ]:
# Parametrize baselines for ALL models (including extended ones)
all_cox_models_dict = {name: ctv for name, (ctv, _) in all_cox_results.items()}
all_baselines = parametrize_all_baselines(all_cox_models_dict)

for name, params in sorted(all_baselines.items()):
    print(f"\n{name}: type={params['type']}")
    if params['type'] == 'linear':
        print(f"  slope (a) = {params['slope']:.6e}")
    else:
        print(f"  shape (k) = {params['shape']:.4f}")
        print(f"  scale (lambda) = {params['scale']:.4f}")
    print(f"  RMSE = {params['rmse']:.6e}")

In [ ]:
# Compute decay-to-baseline times for all Weibull models
decay_times = compute_all_decay_times(all_baselines)

print("Decay-to-baseline times (time for Weibull hazard to reach Model 1 constant level):")
for name, t_star in sorted(decay_times.items()):
    if np.isfinite(t_star):
        print(f"  {name}: t* = {t_star:.1f} hours ({t_star/24:.1f} days)")
    else:
        print(f"  {name}: t* = inf (hazard never decays to baseline)")

In [ ]:
# Plot instantaneous baseline hazards for all models on one figure
plot_instantaneous_baseline_hazards(
    all_baselines, t_max_hours=2000, decay_times=decay_times,
    save_path='instantaneous_baseline_hazards.png',
)
plt.show()

In [ ]:
# Plot decay-to-baseline times as bar chart
plot_decay_times(decay_times, save_path='decay_times.png')
plt.show()

In [ ]:
# Summary table: Model N, event count, Weibull k, Weibull lambda, RMSE, decay time
print(f"{'Model':<12} {'Events':>8} {'Type':<10} {'k':>8} {'lambda':>12} {'RMSE':>12} {'Decay (days)':>14}")
print("-" * 82)
for name in sorted(all_baselines.keys()):
    params = all_baselines[name]
    suffix = name.split("_", 1)[1]
    model_num = int(suffix.rstrip("ab"))
    events = event_counts.get(model_num, "N/A")
    fit_type = params["type"]
    if fit_type == "linear":
        k_str = "---"
        lam_str = f"a={params['slope']:.2e}"
        decay_str = "---"
    else:
        k_str = f"{params['shape']:.4f}"
        lam_str = f"{params['scale']:.1f}"
        t_star = decay_times.get(name, float("inf"))
        decay_str = f"{t_star/24:.1f}" if np.isfinite(t_star) else "inf"
    rmse_str = f"{params['rmse']:.2e}"
    print(f"{name:<12} {str(events):>8} {fit_type:<10} {k_str:>8} {lam_str:>12} {rmse_str:>12} {decay_str:>14}")

## 6. Gateway Effects (Model 2)

In [ ]:
ctv_2b = cox_results['model_2b'][0]
gateway = identify_gateway_conspiracies(ctv_2b)

print("All gateway coefficients (first_conspiracy dummies from Model 2b):")
print(gateway['all'].to_string(index=False))

if 'cross_cluster' in gateway:
    cc = gateway['cross_cluster']
    print(f"\nCross-cluster effect: HR={cc['hazard_ratio']:.3f}, p={cc['p_value']:.4f}")

print(f"\nSignificant accelerators (HR > 1):")
print(gateway['accelerators'][['conspiracy', 'hazard_ratio', 'p_value']].to_string(index=False))

print(f"\nSignificant inhibitors (HR < 1):")
print(gateway['inhibitors'][['conspiracy', 'hazard_ratio', 'p_value']].to_string(index=False))

### Pooled Model: All 2nd+ Adoptions (stratified by adoption order and conspiracy)

Pool Models 2, 3, 4+ into a single Cox fit with `strata=['adoption_order', 'conspiracy']`:

- **Shared coefficients** across adoption orders and conspiracies for tighter CIs on the exposure dose response.
- **Separate baselines per (adoption_order × conspiracy)** so each stage and conspiracy combination gets its own baseline hazard shape.
- **`cross_cluster` remains a covariate** because it is a property of the adoption event, not a grouping factor. It varies within every stratum because different users reach each order and conspiracy combination with different prior adoption paths.

Modeling note: this model assumes the effect of `s7_*`, `degree`, and `cross_cluster` is shared across all adoption orders and conspiracies. Inspect `cox_results` coefficients before relying on the pooled result. Lifelines cannot compute cluster robust SEs for CoxTimeVaryingFitter, so within subject correlation remains an inference caveat.

This specification is **methodologically symmetric with the stratified Model 1 below**: both models use strata on the conspiracy dimension, so Figure 1c compares like-to-like.

In [ ]:
from conspiracy_analysis.data.preprocessing import create_long_form
from lifelines import CoxTimeVaryingFitter


def _prepare_stage_long_form(df_short, G, prior_time_col, adoption_order):
    """Build long form intervals on the time since prior adoption scale."""
    df_long = create_long_form(
        df_short, G, step=STEP,
        correction_for_censored_neighbors=CORRECTION_FOR_CENSORED,
        exposure_window=EXPOSURE_WINDOW_HOURS,
    )
    df_long = df_long.merge(
        df_short[['id', prior_time_col]].drop_duplicates(),
        on='id', how='left',
    )
    df_long['entry'] = (df_long['entry'] - df_long[prior_time_col]).clip(lower=0)
    df_long['exit'] = df_long['exit'] - df_long[prior_time_col]
    df_long = df_long.drop(
        columns=[prior_time_col, 'first_conspiracy'],
        errors='ignore',
    )
    df_long['adoption_order'] = adoption_order
    return df_long


pooled = [
    _prepare_stage_long_form(df_short_2, G, 'first_time', 2),
    _prepare_stage_long_form(df_short_3, G, 'second_time', 3),
]
for name, df_n in extended_short_form.items():
    n = int(name.split('_')[1])
    pooled.append(_prepare_stage_long_form(df_n, G, 'prior_time', n))

df_long_pooled = pd.concat(pooled, ignore_index=True)
df_long_pooled = df_long_pooled[df_long_pooled['exit'] > df_long_pooled['entry']].copy()

expected_min_rows = sum(len(p) for p in pooled)
print(f"Per stage row counts: {[len(p) for p in pooled]}")
print(f"Concatenated rows: {expected_min_rows}; after exit greater than entry filter: {len(df_long_pooled)}")
print(f"\nRows per adoption_order:\n{df_long_pooled['adoption_order'].value_counts().sort_index()}")
print(f"\nEvents per adoption_order:\n{df_long_pooled.groupby('adoption_order')['event'].sum()}")
print("\nEvents per adoption_order and conspiracy:")
event_grid = (
    df_long_pooled.groupby(['adoption_order', 'conspiracy'])['event']
    .sum().unstack(fill_value=0).sort_index()
)
print(event_grid.to_string())
n_empty_strata = int((event_grid == 0).values.sum())
print(f"\nEmpty adoption_order and conspiracy strata: {n_empty_strata} out of {event_grid.size} total")

df_long_pooled['s_7'] = df_long_pooled['s_7'].astype(int)
df_long_pooled['s7_d1'] = (df_long_pooled['s_7'] == 1).astype(int)
df_long_pooled['s7_d2'] = (df_long_pooled['s_7'] == 2).astype(int)
df_long_pooled['s7_d3'] = (df_long_pooled['s_7'] == 3).astype(int)
df_long_pooled['s7_d4'] = (df_long_pooled['s_7'] >= 4).astype(int)
df_long_pooled['degree'] = np.log1p(df_long_pooled['degree'])

formula = 's7_d1 + s7_d2 + s7_d3 + s7_d4 + degree + cross_cluster'

ctv_pooled = CoxTimeVaryingFitter(penalizer=PENALIZER)
ctv_pooled.fit(
    df_long_pooled,
    id_col='id', event_col='event',
    start_col='entry', stop_col='exit',
    strata=['adoption_order', 'conspiracy'],
    formula=formula,
    show_progress=False,
)

print(f"\nPooled model: {df_long_pooled['event'].sum()} events, {len(ctv_pooled.params_)} parameters")
ctv_pooled.print_summary()

joblib.dump(ctv_pooled, intermediate_path('model_pooled_cox.pkl'))
print("\nSaved model_pooled_cox.pkl")


### Stratified Model 1: First Adoption (stratified by conspiracy)

Re-fit Model 1 with `strata=['conspiracy']` to place it on equal methodological
footing with the pooled 2nd+ model above. Each conspiracy gets its own baseline
hazard `h0_c(t)`, and the exposure / degree coefficients are pooled across
conspiracies. This addresses the proportional hazards violations that the
dummy coded Model 1 exhibits for the `fc_ConsProb_*` terms (see §5c).

1. **Used by Figure 2** in `03_main_text_figures.ipynb` as the Model 1 side of the
   Model 1 vs Pooled exposure HR comparison. Both axes use a stratified
   specification for consistent interpretation of the exposure coefficients.
2. **Not used anywhere else.** The dummy coded `model_1_cox.pkl` remains the
   primary artifact for Figure 6 (gateway 2D) and any other analysis that
   requires the per conspiracy `fc_ConsProb_*` coefficients.
3. **Caveat** (same as the pooled model): lifelines' `CoxTimeVaryingFitter`
   does not support cluster robust SEs, so reported CIs are anti conservative
   under within subject correlation.

In [ ]:
from lifelines import CoxTimeVaryingFitter
from conspiracy_analysis.data.preprocessing import create_long_form

# Build long-form for Model 1 on the absolute time axis, same as
# the dummy-coded Model 1. Strata=['conspiracy'] gives each conspiracy
# its own baseline hazard.
df_long_m1_strat = create_long_form(
    df_short_1, G, step=STEP,
    correction_for_censored_neighbors=CORRECTION_FOR_CENSORED,
    exposure_window=EXPOSURE_WINDOW_HOURS,
)

# Feature engineering (mirrors fit_cox_model lines 58-64)
df_long_m1_strat['s_7']   = df_long_m1_strat['s_7'].astype(int)
df_long_m1_strat['s7_d1'] = (df_long_m1_strat['s_7'] == 1).astype(int)
df_long_m1_strat['s7_d2'] = (df_long_m1_strat['s_7'] == 2).astype(int)
df_long_m1_strat['s7_d3'] = (df_long_m1_strat['s_7'] == 3).astype(int)
df_long_m1_strat['s7_d4'] = (df_long_m1_strat['s_7'] >= 4).astype(int)
df_long_m1_strat['degree'] = np.log1p(df_long_m1_strat['degree'])
df_long_m1_strat = df_long_m1_strat[
    df_long_m1_strat['exit'] > df_long_m1_strat['entry']
].copy()

formula_m1_strat = 's7_d1 + s7_d2 + s7_d3 + s7_d4 + degree'

ctv_m1_strat = CoxTimeVaryingFitter(penalizer=PENALIZER)
ctv_m1_strat.fit(
    df_long_m1_strat,
    id_col='id', event_col='event',
    start_col='entry', stop_col='exit',
    strata=['conspiracy'],
    formula=formula_m1_strat,
    show_progress=False,
)

print(f"\nStratified Model 1: {df_long_m1_strat['event'].sum()} events, "
      f"{len(ctv_m1_strat.params_)} parameters")
ctv_m1_strat.print_summary()
joblib.dump(ctv_m1_strat, intermediate_path('model_1_strat_cox.pkl'))
print('Saved model_1_strat_cox.pkl')

### Two-Dimensional Gateway Identification

Combine contagiousness (Model 1 fc_ HRs) with acceleration of second adoption
(Model 2 fc_ HRs) to characterize gateway conspiracies in two dimensions.

In [ ]:
from conspiracy_analysis.models.gateway_effects import identify_gateway_2d
from conspiracy_analysis.visualization.plots import plot_gateway_scatter

gateway_2d = identify_gateway_2d(
    cox_results['model_1'][0],
    cox_results['model_2b'][0],
    reference_conspiracy=BASELINE_CONSPIRACY,
)
print("Two-dimensional gateway characterization:")
print(gateway_2d.to_string(index=False))

plot_gateway_scatter(
    gateway_2d, cluster_assignments=cluster_assignments,
    reference_conspiracy=BASELINE_CONSPIRACY,
)
plt.show()

## 7. Semantic Barrier Analysis

In [ ]:
# Extract user timelines
user_timelines = extract_user_timelines(
    G, cluster_assignments,
    bot_score_threshold=BOT_SCORE_THRESHOLD,
    mode=MODE,
    min_conspiracies=2,
)
print(f"Extracted {len(user_timelines)} user timelines")

In [ ]:
# Cognitive barrier analysis
gaps = compute_semantic_barrier_analysis(user_timelines, cluster_assignments)

print(f"Within-cluster gaps: {len(gaps['within_cluster'])}")
print(f"Between-cluster gaps: {len(gaps['between_clusters'])}")

# Statistical test
barrier_test = test_barrier_significance(gaps)
print(f"\nMann-Whitney U test (within < between): p = {barrier_test['p_value']:.4e}")

# Plot
plot_cognitive_barrier(gaps)
plt.show()

In [ ]:
# Settler effect analysis
settler = compute_settler_effect(user_timelines, cluster_assignments)

print(f"Settler sequences found: {len(settler['pre_jump'])}")

# Statistical tests (Friedman + Wilcoxon signed-rank with Holm-Bonferroni)
settler_test = test_settler_significance(settler)
print(f"\nFriedman test: p = {settler_test['friedman']['p_value']:.4e}")
for pair_name, result in settler_test['pairwise'].items():
    print(f"  {pair_name}: p_raw = {result['p_value_raw']:.4e}, p_corrected = {result['p_value_corrected']:.4e}")

# Plot
plot_settler_effect(settler, save_path=f'settler_effect_{MODE.lower()}.png')
plt.show()

## 8. Temporal Emergence Clustering

Compare barrier/settler effects when conspiracies are clustered by their temporal
order of first appearance in the network (rather than semantic distance).

In [ ]:
# Compute temporal distance matrix from first-appearance times
df_temporal = compute_temporal_distance_matrix(G)
print(f"Temporal distance matrix shape: {df_temporal.shape}")

# Print first appearance times for reference
first_times = get_first_appearance_times(G)
for c, t in sorted(first_times.items(), key=lambda x: x[1]):
    print(f"  {c}: hour {t}")

In [ ]:
temporal_clustering = find_optimal_clustering(
    distance_matrix=df_temporal,
    conspiracies=conspiracies,
    methods=['ward', 'average', 'complete', 'single'],
    k_range=range(3, 9),
)

print(f"Best method: {temporal_clustering['best_method']}")
print(f"Best k: {temporal_clustering['best_k']}")
print(f"Silhouette Score: {temporal_clustering['best_score']:.4f}")
print(f"\nCluster assignments:")
for cid, members in sorted(temporal_clustering['cluster_map'].items()):
    print(f"  Cluster {cid}: {', '.join(members)}")

In [ ]:
plot_dendrogram(
    temporal_clustering['linkage_matrix'],
    temporal_clustering['conspiracies'],
    title='Temporal Emergence Distance Dendrogram',
)
plt.show()

In [ ]:
plot_silhouette_heatmap(temporal_clustering['all_scores'])
plt.show()

In [ ]:
temporal_cluster_assignments = assign_clusters(temporal_clustering)

temporal_timelines = extract_user_timelines(
    G, temporal_cluster_assignments,
    bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE, min_conspiracies=2,
)
print(f"Extracted {len(temporal_timelines)} user timelines")

temporal_gaps = compute_semantic_barrier_analysis(temporal_timelines, temporal_cluster_assignments)
print(f"Within-cluster gaps: {len(temporal_gaps['within_cluster'])}")
print(f"Between-cluster gaps: {len(temporal_gaps['between_clusters'])}")

temporal_barrier_test = test_barrier_significance(temporal_gaps)
print(f"\nMann-Whitney U test (within < between): p = {temporal_barrier_test['p_value']:.4e}")

plot_cognitive_barrier(temporal_gaps)
plt.title("Temporal Clustering: Jumping to a New Cluster is Slower\n(Median with 95% Bootstrap CI)")
plt.show()

In [ ]:
temporal_settler = compute_settler_effect(temporal_timelines, temporal_cluster_assignments)
print(f"Settler sequences found: {len(temporal_settler['pre_jump'])}")

temporal_settler_test = test_settler_significance(temporal_settler)
print(f"\nFriedman test: p = {temporal_settler_test['friedman']['p_value']:.4e}")
for pair_name, result in temporal_settler_test['pairwise'].items():
    print(f"  {pair_name}: p_raw = {result['p_value_raw']:.4e}, p_corrected = {result['p_value_corrected']:.4e}")

plot_settler_effect(temporal_settler)
plt.show()

## 9. Peak-Frequency Temporal Clustering

Compare barrier/settler effects when conspiracies are clustered by the time
of their peak sharing frequency (24-hour rolling window) rather than semantic
distance or first appearance.

In [ ]:
df_peak_freq = compute_peak_frequency_distance_matrix(G, window=24)
print(f"Peak-frequency distance matrix shape: {df_peak_freq.shape}")

peak_times = get_peak_frequency_times(G, window=24)
for c, t in sorted(peak_times.items(), key=lambda x: x[1]):
    print(f"  {c}: peak at hour {t}")

In [ ]:
peak_freq_clustering = find_optimal_clustering(
    distance_matrix=df_peak_freq,
    conspiracies=conspiracies,
    methods=['ward', 'average', 'complete', 'single'],
    k_range=range(3, 9),
)
print(f"Best method: {peak_freq_clustering['best_method']}")
print(f"Best k: {peak_freq_clustering['best_k']}")
print(f"Silhouette Score: {peak_freq_clustering['best_score']:.4f}")
print(f"\nCluster assignments:")
for cid, members in sorted(peak_freq_clustering['cluster_map'].items()):
    print(f"  Cluster {cid}: {', '.join(members)}")

In [ ]:
plot_dendrogram(
    peak_freq_clustering['linkage_matrix'],
    peak_freq_clustering['conspiracies'],
    title='Peak-Frequency Distance Dendrogram',
)
plt.show()

In [ ]:
plot_silhouette_heatmap(peak_freq_clustering['all_scores'])
plt.show()

In [ ]:
peak_freq_cluster_assignments = assign_clusters(peak_freq_clustering)

peak_freq_timelines = extract_user_timelines(
    G, peak_freq_cluster_assignments,
    bot_score_threshold=BOT_SCORE_THRESHOLD, mode=MODE, min_conspiracies=2,
)
print(f"Extracted {len(peak_freq_timelines)} user timelines")

peak_freq_gaps = compute_semantic_barrier_analysis(peak_freq_timelines, peak_freq_cluster_assignments)
print(f"Within-cluster gaps: {len(peak_freq_gaps['within_cluster'])}")
print(f"Between-cluster gaps: {len(peak_freq_gaps['between_clusters'])}")

peak_freq_barrier_test = test_barrier_significance(peak_freq_gaps)
print(f"\nMann-Whitney U test (within < between): p = {peak_freq_barrier_test['p_value']:.4e}")

plot_cognitive_barrier(peak_freq_gaps)
plt.title("Peak-Frequency Clustering: Jumping to a New Cluster\n(Median with 95% Bootstrap CI)")
plt.show()

In [ ]:
peak_freq_settler = compute_settler_effect(peak_freq_timelines, peak_freq_cluster_assignments)
print(f"Settler sequences found: {len(peak_freq_settler['pre_jump'])}")

peak_freq_settler_test = test_settler_significance(peak_freq_settler)
print(f"\nFriedman test: p = {peak_freq_settler_test['friedman']['p_value']:.4e}")
for pair_name, result in peak_freq_settler_test['pairwise'].items():
    print(f"  {pair_name}: p_raw = {result['p_value_raw']:.4e}, p_corrected = {result['p_value_corrected']:.4e}")

plot_settler_effect(peak_freq_settler)
plt.show()

In [ ]:
# Bootstrap inference artifacts


def _params_formula(ctv):
    return " + ".join(str(term) for term in ctv.params_.index)


cox_model_objects = {name: ctv for name, (ctv, _) in cox_results.items()}
cox_model_objects["model_1_strat"] = ctv_m1_strat
cox_model_objects["model_pooled"] = ctv_pooled

bootstrap_specs = []
for name, (ctv, df_long) in sorted(cox_results.items()):
    baseline_role = None
    if name == "model_1":
        baseline_role = "linear"
    elif name != "model_2b":
        baseline_role = "weibull"
    bootstrap_specs.append(
        ModelBootstrapSpec(
            name=name,
            df_long=df_long,
            formula=_params_formula(ctv),
            penalizer=PENALIZER,
            baseline_role=baseline_role,
        )
    )

bootstrap_specs.extend([
    ModelBootstrapSpec(
        name="model_1_strat",
        df_long=df_long_m1_strat,
        formula=formula_m1_strat,
        penalizer=PENALIZER,
        strata=["conspiracy"],
    ),
    ModelBootstrapSpec(
        name="model_pooled",
        df_long=df_long_pooled,
        formula=formula,
        penalizer=PENALIZER,
        strata=["adoption_order", "conspiracy"],
    ),
])

master_user_ids = sorted(df_long_m1_strat["id"].unique())

print(f"Running Cox bootstrap smoke validation with {BOOTSTRAP_SMOKE_DRAWS} draws")
cox_bootstrap_smoke = run_cox_user_bootstrap(
    bootstrap_specs,
    master_user_ids=master_user_ids,
    n_draws=BOOTSTRAP_SMOKE_DRAWS,
    seed=BOOTSTRAP_SEED,
    eval_times_hours=BOOTSTRAP_EVAL_TIMES,
    n_jobs=BOOTSTRAP_N_JOBS,
)
print("Smoke draw successes:")
print(cox_bootstrap_smoke["n_success"])
log_bootstrap_success_summary(
    "cox_smoke",
    n_draws=BOOTSTRAP_SMOKE_DRAWS,
    n_jobs=BOOTSTRAP_N_JOBS,
    successes=cox_bootstrap_smoke["n_success"],
)

print(f"Running final Cox bootstrap with {COX_BOOTSTRAP_FINAL_DRAWS} draws")
cox_bootstrap_raw = run_cox_user_bootstrap(
    bootstrap_specs,
    master_user_ids=master_user_ids,
    n_draws=COX_BOOTSTRAP_FINAL_DRAWS,
    seed=BOOTSTRAP_SEED,
    eval_times_hours=BOOTSTRAP_EVAL_TIMES,
    min_successes=BOOTSTRAP_MIN_SUCCESSES,
    n_jobs=BOOTSTRAP_N_JOBS,
)
cox_bootstrap_results = summarize_cox_bootstrap(
    cox_bootstrap_raw,
    cox_models=cox_model_objects,
    baseline_params=all_baselines,
    reference_conspiracy=BASELINE_CONSPIRACY,
)
joblib.dump(cox_bootstrap_results, intermediate_path('cox_bootstrap_results.pkl'))
print("Saved cox_bootstrap_results.pkl")
print("Final draw successes:")
print(cox_bootstrap_results["n_success"])
log_bootstrap_success_summary(
    "cox_final",
    n_draws=COX_BOOTSTRAP_FINAL_DRAWS,
    n_jobs=BOOTSTRAP_N_JOBS,
    successes=cox_bootstrap_results["n_success"],
    min_successes=BOOTSTRAP_MIN_SUCCESSES,
)

timeline_specs = {
    "semantic": (user_timelines, cluster_assignments),
    "temporal": (temporal_timelines, temporal_cluster_assignments),
    "peak_frequency": (peak_freq_timelines, peak_freq_cluster_assignments),
}

print(f"Running timeline bootstrap smoke validation with {BOOTSTRAP_SMOKE_DRAWS} draws")
timeline_bootstrap_smoke = run_timeline_bootstrap(
    timeline_specs,
    n_draws=BOOTSTRAP_SMOKE_DRAWS,
    seed=BOOTSTRAP_SEED,
    n_jobs=BOOTSTRAP_N_JOBS,
)
print(timeline_bootstrap_smoke["intervals"].head())
log_bootstrap_success_summary(
    "timeline_smoke",
    n_draws=BOOTSTRAP_SMOKE_DRAWS,
    n_jobs=BOOTSTRAP_N_JOBS,
    successes={family: BOOTSTRAP_SMOKE_DRAWS for family in timeline_specs},
)

print(f"Running final timeline bootstrap with {TIMELINE_BOOTSTRAP_FINAL_DRAWS} draws")
timeline_bootstrap_results = run_timeline_bootstrap(
    timeline_specs,
    n_draws=TIMELINE_BOOTSTRAP_FINAL_DRAWS,
    seed=BOOTSTRAP_SEED,
    n_jobs=BOOTSTRAP_N_JOBS,
)
joblib.dump(timeline_bootstrap_results, intermediate_path('timeline_bootstrap_results.pkl'))
print("Saved timeline_bootstrap_results.pkl")
log_bootstrap_success_summary(
    "timeline_final",
    n_draws=TIMELINE_BOOTSTRAP_FINAL_DRAWS,
    n_jobs=BOOTSTRAP_N_JOBS,
    successes={family: TIMELINE_BOOTSTRAP_FINAL_DRAWS for family in timeline_specs},
)


In [ ]:
print("=" * 88)
print("Semantic vs. First-Appearance vs. Peak-Frequency Clustering Comparison")
print("=" * 88)
print(f"{'Metric':<30} {'Semantic':>18} {'First-Appear.':>18} {'Peak-Freq.':>18}")
print("-" * 88)
print(f"{'Best method':<30} {clustering_result['best_method']:>18} {temporal_clustering['best_method']:>18} {peak_freq_clustering['best_method']:>18}")
print(f"{'Best k':<30} {clustering_result['best_k']:>18} {temporal_clustering['best_k']:>18} {peak_freq_clustering['best_k']:>18}")
print(f"{'Silhouette Score':<30} {clustering_result['best_score']:>18.4f} {temporal_clustering['best_score']:>18.4f} {peak_freq_clustering['best_score']:>18.4f}")
print(f"{'Barrier p-value':<30} {barrier_test['p_value']:>18.4e} {temporal_barrier_test['p_value']:>18.4e} {peak_freq_barrier_test['p_value']:>18.4e}")
print(f"{'Settler Friedman p':<30} {settler_test['friedman']['p_value']:>18.4e} {temporal_settler_test['friedman']['p_value']:>18.4e} {peak_freq_settler_test['friedman']['p_value']:>18.4e}")

## 10. Summary

Key findings from the analysis:
- **Model 1**: First conspiracy adoption is driven by neighbor exposure (s7 dummies) and network degree.
- **Model 2**: Second conspiracy adoption is accelerated by prior adoption, with gateway effects from specific first conspiracies.
- **Model 3+**: Third+ adoption continues to accelerate, with cross-cluster transitions showing a significant barrier effect.
- **Semantic clustering**: Optimal clustering found via Silhouette Score optimization.
- **Cognitive barrier**: Within-cluster transitions are significantly faster than between-cluster transitions.
- **Settler effect**: Users show a characteristic slow-down when crossing cluster boundaries, followed by rapid settling in the new cluster.